# Audit model konteks pada anotasi manual train/validation

Buka dari root **SongMap**. Notebook menjalankan checkpoint konteks epoch 5 yang sudah dipakai aplikasi pada **cache section yang sudah dibuat saat training**. Anotasi manual dan split train/val tetap. Section held/dikecualikan tidak menjadi target.

Tidak ada encoder MuQ/MusicFM, SongFormer head lama, decoding audio, ASR atau optimizer yang dijalankan. Hanya head konteks kecil pada cache ringkasan. Cache yang hilang/rusak membuat pemeriksaan berhenti; notebook tidak memulai ekstraksi ulang. Environment Python di folder Section-Annotations dipakai secara eksplisit.

Tujuannya: mengetahui pola kesalahan pada banyak lagu—bridge ↔ chorus, fungsi vokal ↔ inst, confidence, posisi dan regresi setelah context. Ini audit label terhadap anotasi yang ada, bukan penilaian boundary independen. Hasil train dan validation dipisahkan.


In [ ]:
from pathlib import Path
from datetime import datetime
import json,subprocess

ROOT=Path.cwd().resolve()
assert (ROOT/'scripts/audit_manual_context.py').is_file(), 'Buka dari root SongMap'
SONGSTRUCT_ROOT=ROOT.parent/'Section-Annotations'
RUN_PYTHON=SONGSTRUCT_ROOT/'.venv-songformer311/bin/python'  # Pertahankan path venv, jangan resolve symlink Python.
assert RUN_PYTHON.is_file(), f'Python tidak ditemukan: {RUN_PYTHON}'
CACHE=ROOT/'outputs.local/section-context-cache-v1'
assert CACHE.is_dir(), 'Cache section dari training sebelumnya belum ditemukan; periksa CACHE'
receipt=ROOT/'SongMap/Resources/MLModels/SongFormerContext/songformer_section_context_manifest.json'
EXPECTED_CONTEXT_SHA=(json.loads(receipt.read_text())['checkpointSHA256'] if receipt.is_file()
                      else '1511a8c4c53b86a3c58508a4a7c2f890daa1adb926acbdb9dae56ad529d01e01')
candidates=[]
for path in (ROOT/'outputs.local').glob('frozen-context-test-*/selection.json'):
    lock=json.loads(path.read_text())
    if lock.get('context_epoch')==5 and lock.get('context_checkpoint_sha256')==EXPECTED_CONTEXT_SHA and (path.parent/'frozen-context.pt').is_file():
        candidates.append(path)
assert candidates, 'Snapshot epoch 5 yang cocok belum ditemukan; periksa folder frozen-context-test-*'
FROZEN=max(candidates,key=lambda p:p.stat().st_mtime).parent
OUTPUT=ROOT/'outputs.local'/('manual-context-audit-'+datetime.now().strftime('%Y%m%d-%H%M%S-%f'))
subprocess.run([str(RUN_PYTHON),'-c','import sys,numpy,torch; print("Python:",sys.executable); print("Torch:",torch.__version__)'],check=True)
print('Model terkunci:',FROZEN)
print('Cache section:',CACHE)
print('Hasil:',OUTPUT)

def run_audit(limit,name):
    command=[str(RUN_PYTHON),'-u',str(ROOT/'scripts/audit_manual_context.py'),
             '--songstruct-root',str(SONGSTRUCT_ROOT),'--frozen-evaluation',str(FROZEN),
             '--cache-dir',str(CACHE),'--output-dir',str(OUTPUT),'--limit',str(limit)]
    log=Path(str(OUTPUT)+'.'+name+'.log');log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('w',encoding='utf-8') as handle:
        process=subprocess.Popen(command,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:raise RuntimeError(f'Audit gagal ({status}); lihat {log}. Cache/prediksi yang selesai tetap tersimpan.')


In [ ]:
# Satu train + satu validation. Belum mewakili seluruh dataset.
run_audit(1,'smoke')


In [ ]:
# Seluruh train + validation; hasil smoke dipakai ulang bila identitasnya cocok.
run_audit(0,'all')


In [ ]:
from IPython.display import display,Markdown
report=json.loads((OUTPUT/'audit-all.json').read_text())
assert report['full_train_val'] and report['original_context_validation_reproduced']
lines=['| Split | Lagu | Section | V3 acc | Context acc | V3 F1 | Context F1 | Salah→benar | Benar→salah |',
       '|---|---|---|---|---|---|---|---|---|']
for split,r in report['splits'].items():
    b=r['baseline'];c=r['context']
    lines.append(f"| {split} | {r['tracks']} | {c['sections']} | {b['section_accuracy']:.2%} | {c['section_accuracy']:.2%} | {b['macro_f1_present']:.4f} | {c['macro_f1_present']:.4f} | {r['wrong_to_correct']} | {r['correct_to_wrong']} |")
display(Markdown('\n'.join(lines)))
for split,r in report['splits'].items():
    print('\n',split,'pola kesalahan context:',r['context_error_patterns'])
    print('Confusion teratas:',r['context_confusion_pairs'][:8])
    print('Bridge menurut posisi:',r['manual_bridge_predictions_by_position'])
native=report['saved_native_palette6_validation']
print('\nSaved native palette6 validation:',native['status'])
if native['status']=='saved_metrics_reproduced':
    print('Original-feature context F1:',native['original_feature_context']['macro_f1_present'])
    print('Saved native/palette6 F1:',native['saved_native_context']['macro_f1_present'])
    print('Pola native:',native['error_patterns'])
print('\nKirim:',OUTPUT/'audit-all.json')
print('Daftar error:',OUTPUT/'errors-all.csv')


Hasil utama memakai **encoder cache asli saat training**, sehingga terpisah dari prediksi audio-ke-label iPhone yang terkompresi. Jika laporan paritas Core ML lengkap yang cocok tersedia, script juga membaca **prediksi validation native/palette6 yang sudah tersimpan** dan mereproduksi metriknya. Tidak ada inferensi Core ML baru. Perbandingan ini dilaporkan terpisah agar perbedaan encoder tidak disamakan dengan efek training.

Anotasi lama tetap menjadi rujukan; probability model yang tinggi tidak membuktikan anotasi salah. Penilaian test tidak dijalankan. Metadata split digunakan untuk memeriksa kebocoran grup/audio, bukan menghitung skor test. Validation juga pernah dipakai memilih model sebelumnya, sehingga hasilnya bukan estimasi final baru.

Kirim `audit-all.json` setelah selesai. Tidak perlu melabeli ulang lagu dari CSV: CSV berisi daftar perbedaan untuk diagnosis, bukan tugas anotasi baru. Berdasarkan pola train/val inilah eksperimen finetuning berikutnya dipilih; belum ada model baru atau aturan label yang diaktifkan oleh notebook ini.
